# Polynomial Features: Capturing Non-Linear Relationships

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/03-feature-engineering/04_polynomial_features.ipynb)

## Objectives
- Create polynomial features from numeric variables
- Generate interaction terms
- Understand bias-variance tradeoff with polynomials
- Choose optimal polynomial degree
- Prevent overfitting with high-degree polynomials

## 1. Why Polynomial Features?

**Motivation:**
- Linear models can't capture non-linear relationships
- Polynomial features make linear models more flexible
- Example: y = ax³ + bx² + cx + d looks linear in polynomial space

**Trade-offs:**
- ✅ Can fit complex non-linear relationships
- ✅ Simple to implement and interpret
- ❌ Risk of overfitting (curse of dimensionality)
- ❌ Creates many features from few inputs
- ❌ Computational cost increases with degree

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
sns.set_theme()

print("✅ Libraries loaded")

# Generate data with cubic relationship: y = 2x^3 - 5x^2 + 3x + noise
n_samples = 200
X = np.linspace(-3, 3, n_samples).reshape(-1, 1)
y = 2 * X**3 - 5 * X**2 + 3 * X + np.random.normal(0, 10, (n_samples, 1))
y = y.ravel()

df = pd.DataFrame({'x': X.ravel(), 'y': y})

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head(10))

# Visualization
plt.figure(figsize=(12, 5))
plt.scatter(df['x'], df['y'], alpha=0.6, s=50)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Non-Linear Relationship: y = 2x³ - 5x² + 3x + noise')
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: Clear non-linear cubic pattern in data")

# Generate polynomial features
poly_features = PolynomialFeatures(degree=3, include_bias=False)
X_poly = poly_features.fit_transform(X)

# Create DataFrame to see feature names
poly_df = pd.DataFrame(
    X_poly,
    columns=poly_features.get_feature_names_out(['x'])
)

print("📊 Polynomial Features (Degree 3):")
print(f"Original features: 1 (just 'x')")
print(f"Polynomial features: {X_poly.shape[1]} (x, x², x³)")
print(f"\nFeature names generated:")
print(poly_features.get_feature_names_out(['x']).tolist())
print(f"\nFirst 10 rows of polynomial features:")
print(poly_df.head(10))

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model 1: Simple Linear Regression
lr_linear = LinearRegression()
lr_linear.fit(X_train, y_train)
y_pred_linear = lr_linear.predict(X_test)
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

# Model 2: Polynomial (degree=3)
poly_transformer = PolynomialFeatures(degree=3, include_bias=False)
X_train_poly = poly_transformer.fit_transform(X_train)
X_test_poly = poly_transformer.transform(X_test)

lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)
mse_poly = mean_squared_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print("📊 Model Comparison:")
print(f"\nLinear Regression:")
print(f"  MSE: {mse_linear:.2f}")
print(f"  R²: {r2_linear:.4f}")

print(f"\nPolynomial Regression (degree=3):")
print(f"  MSE: {mse_poly:.2f}")
print(f"  R²: {r2_poly:.4f}")

print(f"\nImprovement:")
print(f"  MSE improved by: {((mse_linear - mse_poly) / mse_linear * 100):.1f}%")
print(f"  R² improved by: {((r2_poly - r2_linear) / abs(r2_linear) * 100):.1f}%")

# Create smooth curves for visualization
X_smooth = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_linear_smooth = lr_linear.predict(X_smooth)
X_smooth_poly = poly_transformer.transform(X_smooth)
y_poly_smooth = lr_poly.predict(X_smooth_poly)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Linear fit
axes[0].scatter(X_test, y_test, alpha=0.6, s=50, label='Test Data', color='blue')
axes[0].plot(X_smooth, y_linear_smooth, 'r-', linewidth=2, label='Linear Fit')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title(f'Linear Regression (R² = {r2_linear:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Polynomial fit
axes[1].scatter(X_test, y_test, alpha=0.6, s=50, label='Test Data', color='blue')
axes[1].plot(X_smooth, y_poly_smooth, 'g-', linewidth=2, label='Polynomial Fit (degree=3)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].set_title(f'Polynomial Regression (R² = {r2_poly:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test different polynomial degrees
degrees = range(1, 11)
train_scores = []
test_scores = []
test_mses = []

for degree in degrees:
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_p = poly.fit_transform(X_train)
    X_test_p = poly.transform(X_test)
    
    # Train model
    model = LinearRegression()
    model.fit(X_train_p, y_train)
    
    # Score
    train_score = model.score(X_train_p, y_train)
    test_score = model.score(X_test_p, y_test)
    test_mse = mean_squared_error(y_test, model.predict(X_test_p))
    
    train_scores.append(train_score)
    test_scores.append(test_score)
    test_mses.append(test_mse)

# Results
results_df = pd.DataFrame({
    'Degree': list(degrees),
    'Train R²': train_scores,
    'Test R²': test_scores,
    'Test MSE': test_mses
})

print("📊 Polynomial Degree Performance:")
print(results_df.to_string(index=False))

# Find optimal degree (highest test R²)
optimal_degree = results_df.loc[results_df['Test R²'].idxmax(), 'Degree']
print(f"\n🎯 Optimal degree: {int(optimal_degree)} (Test R² = {results_df['Test R²'].max():.4f})")

# Plot training vs test performance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² scores
axes[0].plot(degrees, train_scores, 'o-', label='Train R²', linewidth=2, markersize=8, color='blue')
axes[0].plot(degrees, test_scores, 's-', label='Test R²', linewidth=2, markersize=8, color='red')
axes[0].axvline(optimal_degree, color='green', linestyle='--', label=f'Optimal (degree={int(optimal_degree)})')
axes[0].set_xlabel('Polynomial Degree')
axes[0].set_ylabel('R² Score')
axes[0].set_title('Bias-Variance Tradeoff: Train vs Test Performance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MSE
axes[1].plot(degrees, test_mses, 'o-', linewidth=2, markersize=8, color='red')
axes[1].axvline(optimal_degree, color='green', linestyle='--', label=f'Optimal (degree={int(optimal_degree)})')
axes[1].set_xlabel('Polynomial Degree')
axes[1].set_ylabel('Test MSE')
axes[1].set_title('Test Error vs Polynomial Degree')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n⚠️ Notice: High degrees cause overfitting!")
print(f"   Degree 1: Train R²={train_scores[0]:.4f}, Test R²={test_scores[0]:.4f}")
print(f"   Degree 9: Train R²={train_scores[8]:.4f}, Test R²={test_scores[8]:.4f}")
print(f"   Gap at degree 9: {train_scores[8] - test_scores[8]:.4f} (huge overfitting!)")

# Create dataset with multiple features
n_samples = 300
X_multi = np.random.randn(n_samples, 2) * 2  # 2 features

# Target: y = 2*x1 + 3*x2 + 5*x1*x2 + noise (includes interaction)
y_multi = 2 * X_multi[:, 0] + 3 * X_multi[:, 1] + 5 * X_multi[:, 0] * X_multi[:, 1] + np.random.normal(0, 5, n_samples)

# Generate interaction features
poly_interact = PolynomialFeatures(degree=2, include_bias=False)
X_interact = poly_interact.fit_transform(X_multi)

print("📊 Interaction Features (Degree 2):")
print(f"Original features: 2")
print(f"With interactions: {X_interact.shape[1]}")
print(f"Features: x1, x2, x1², x1*x2, x2²")

# Compare models
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)
X_train_int, X_test_int, _, _ = train_test_split(X_interact, y_multi, test_size=0.2, random_state=42)

# Model without interactions
model_no_int = LinearRegression()
model_no_int.fit(X_train_m, y_train_m)
r2_no_int = model_no_int.score(X_test_m, y_test_m)

# Model with interactions
model_with_int = LinearRegression()
model_with_int.fit(X_train_int, y_train_m)
r2_with_int = model_with_int.score(X_test_int, y_test_m)

print(f"\n📊 Model Performance:")
print(f"Without interactions: R² = {r2_no_int:.4f}")
print(f"With interactions: R² = {r2_with_int:.4f}")
print(f"Improvement: {(r2_with_int - r2_no_int) * 100:.2f}%")

print(f"\nCoefficients (with interactions):")
print(f"x1: {model_with_int.coef_[0]:.4f} (true: 2)")
print(f"x2: {model_with_int.coef_[1]:.4f} (true: 3)")
print(f"x1²: {model_with_int.coef_[2]:.4f} (true: 0)")
print(f"x1*x2: {model_with_int.coef_[3]:.4f} (true: 5) ✓ Interaction captured!")
print(f"x2²: {model_with_int.coef_[4]:.4f} (true: 0)")

# Use Ridge regression with different alpha values
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_train_scores = []
ridge_test_scores = []

# Use degree 9 polynomials (prone to overfitting)
poly_high = PolynomialFeatures(degree=9, include_bias=False)
X_train_high = poly_high.fit_transform(X_train)
X_test_high = poly_high.transform(X_test)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_high, y_train)
    ridge_train_scores.append(ridge.score(X_train_high, y_train))
    ridge_test_scores.append(ridge.score(X_test_high, y_test))

# Plot
plt.figure(figsize=(12, 5))
plt.plot(alphas, ridge_train_scores, 'o-', label='Train R²', linewidth=2, markersize=8)
plt.plot(alphas, ridge_test_scores, 's-', label='Test R²', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('Ridge Alpha (Regularization Strength)')
plt.ylabel('R² Score')
plt.title('Ridge Regression: Controlling Overfitting with Regularization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📊 Ridge Regularization Results (Degree 9):")
for i, alpha in enumerate(alphas):
    print(f"Alpha={alpha:>6.3f}: Train R²={ridge_train_scores[i]:.4f}, Test R²={ridge_test_scores[i]:.4f}, Gap={ridge_train_scores[i]-ridge_test_scores[i]:.4f}")

print("""\n📋 POLYNOMIAL FEATURES BEST PRACTICES:

✅ DO:
   1. Start with low degree (2-3) and increase gradually
   2. Use cross-validation to select optimal degree
   3. Apply regularization (Ridge, Lasso) with high degrees
   4. Scale features before creating polynomials
   5. Use domain knowledge to create specific interactions
   6. Monitor test performance (not just training)

❌ DON'T:
   1. Use very high degrees (>5) without regularization
   2. Create polynomial features for tree-based models
   3. Forget to apply transformations to test data
   4. Use all polynomial interactions (~n² features from n predictors)
   5. Ignore multicollinearity (polynomial features are highly correlated)

⚠️  OVERFITTING SIGNS:
   • Large gap between train and test R²
   • Train accuracy keeps improving but test plateaus
   • Model coefficients become very large

🔧 SOLUTIONS:
   • Lower polynomial degree
   • Use L1/L2 regularization
   • Collect more data
   • Feature selection (remove least important terms)
""")

print("""\n📚 KEY TAKEAWAYS:

Polynomial Features:
✓ Transform non-linear relationships to linear space
✓ Degree 2-3 usually sufficient (rarely need > 5)
✓ Always use cross-validation for degree selection
✓ Combine with regularization for high degrees
✓ Not needed for tree-based models

Trade-offs:
✓ Higher degree = Better fit to training data
✓ Higher degree = Higher overfitting risk
✓ Find sweet spot using test performance

Next Steps:
→ Binning & discretization (continuous → categorical)
→ DateTime feature extraction
→ Text feature engineering""")